# 02 — Load, clean and prepare offline data

Builds the network-free corpora that notebook 03 and `sandbox datasets prepare` use.

| Source | Input | Output under `data/runtime/prepared/` |
| --- | --- | --- |
| Fixture catalog | `data/fixtures/manifest.csv` + the text files it names | `fixtures_prepared.jsonl` |
| HF mini slice | `data/fixtures/hf/docclass_mini.jsonl` | `hf_prepared.jsonl` |
| Agent gold | `data/fixtures/agents/*.jsonl` | `agents/<agent>_prepared.jsonl` |
| Sorter pilot | first fixture of each document class | `sorter_pilot_balanced.jsonl` |
| Run summary | all of the above | `prep_manifest.json` |

**Cleaning rules:** CRLF → LF, runs of spaces and tabs → one space, three or more newlines → one blank line, and trimmed ends. A row is **dropped** if a required key is missing, its file is missing, or it is empty after cleaning. A malformed `expected_fields` value is logged and replaced with `{}`.

**Cost:** none. Tracked fixtures are only read. Writes go to `data/runtime/`, which is gitignored. Rerunning gives the same fingerprint.

**Prerequisite:** notebook [01](01_offline_environment_setup.ipynb), or an existing `.env` and a profile that activates.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Works with or without `pip install -e .`: fall back to the checkout's src/ when needed.
try:
    import mailroom_sandbox  # noqa: F401
except ModuleNotFoundError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "src" / "mailroom_sandbox").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import mailroom_sandbox  # noqa: F401

from mailroom_sandbox.notebook import bootstrap

ctx = bootstrap()
print(json.dumps(ctx.as_dict(), indent=2))

In [ ]:
try:
    import pandas as pd
except ImportError as exc:  # pragma: no cover - guidance for a bare [dev] install
    raise ImportError('pandas is missing: pip install -e ".[dev,notebooks]"') from exc

pd.set_option("display.max_colwidth", 80)

## 1. The raw fixture catalog

`load_manifest()` reads `data/fixtures/manifest.csv` as is. Each row names a fixture file by `subdir` and `filename`, and carries the expected class, the expected pipeline stage and, optionally, gold fields as JSON.

In [ ]:
from mailroom_sandbox.datasets import load_manifest

raw = load_manifest()
raw_df = pd.DataFrame(raw)
print(f"{len(raw_df)} catalog rows, {raw_df['expected_doc_class'].nunique()} classes")
display(raw_df[["id", "subdir", "filename", "expected_doc_class", "expected_stage", "size_tier"]])

In [ ]:
pd.concat(
    {
        "class": raw_df["expected_doc_class"].value_counts(),
        "stage": raw_df["expected_stage"].replace("", "(none)").value_counts(),
    }
).rename("rows").to_frame()

## 2. What `normalize_text` does

A quick before-and-after example on a messy string. Paragraph breaks are kept. Everything else collapses.

In [ ]:
from mailroom_sandbox.prep import normalize_text

messy = "  MASTER   SERVICES\tAGREEMENT\r\n\r\n\r\n\r\nThis  Agreement is made…  \n\n\n"
print("before:", repr(messy))
print("after: ", repr(normalize_text(messy)))

## 3. Clean the fixture catalog and the HF slice

Each cleaner returns `(rows, CleanReport)`. The report counts what was kept, dropped or repaired, and lists one `issues` entry per problem row, so review it before trusting the output.

In [ ]:
from mailroom_sandbox.prep import clean_hf_rows, clean_manifest_rows

fixtures, fix_report = clean_manifest_rows()
hf_rows, hf_report = clean_hf_rows()

reports_df = pd.DataFrame([fix_report.as_dict(), hf_report.as_dict()]).set_index("source")
display(reports_df.drop(columns="issues"))
for report in (fix_report, hf_report):
    for issue in report.issues:
        print(f"[{report.source}] {issue}")

assert fix_report.kept_rows > 0, "no fixture rows survived cleaning"
assert hf_report.kept_rows > 0, "the HF mini slice is empty"

In [ ]:
fixtures_df = pd.DataFrame(fixtures)
display(fixtures_df[["id", "expected_doc_class", "expected_stage", "char_count", "path"]])

first = fixtures[0]
print(f"\n{first['id']} ({first['expected_doc_class']}, {first['char_count']} chars):")
print(first["text"][:400], "…")
print("\nexpected_fields:", json.dumps(first["expected_fields"], indent=2)[:400])

In [ ]:
hf_df = pd.DataFrame(hf_rows)
display(hf_df[["id", "doc_type", "expected_hf_class", "split", "char_count"]])

### Is the gold grounded in the text?

`gold_field_grounding()` checks that each extractive gold value in `expected_fields`, such as a party name or a governing law, appears in the cleaned text (ignoring case and whitespace runs), literally or in a canonical form: an ISO date, an amount by value, a spelled-out numeral or an alias such as `SEC`. Those matches are `grounded_normalized` and listed with the form that matched. A value that is **ungrounded** usually means the gold labels belong to another document. Mock evals echo the gold labels, so only a live run would otherwise expose it. Categorical labels (`communication_type`, `merger_consideration`) are classifications, not quotes, and booleans are not text; both are counted but not checked. This cell is a gate: fix any ungrounded row in `data/fixtures/manifest.csv`.

In [ ]:
# Every extractive gold value must be written in its fixture text (case/whitespace-insensitive),
# literally or in a canonical form: ISO dates, amounts by value, spelled-out numerals.
# Categorical labels (communication_type, merger_consideration) are classifications, not spans.
from mailroom_sandbox.notebook import (
    agent_fixture_grounding,
    cached_doc_rows,
    gold_field_grounding,
    label_overlay_grounding,
    merged_truth,
    ungrounded,
)

grounding = gold_field_grounding(fixtures) + agent_fixture_grounding()
print(Counter(f.status for f in grounding))
for f in grounding:
    if f.status == "grounded_normalized":
        print(f"normalized ({f.form}) {f.row_id}.{f.field}: {f.value!r}")
missing = ungrounded(grounding)
for f in missing:
    print(f"UNGROUNDED {f.row_id}.{f.field}: {f.value!r}")
assert not missing, "gold values not found in fixture text; fix data/fixtures/manifest.csv or data/fixtures/agents/"

# Scoring overlays (near-gold, interim): each truth label checked by the rule that admitted it
# (stored quote, review rule, derivation, vocabulary). With a pulled Hub corpus
# (`sandbox datasets pull`), labels that stored no quote are checked against their document too.
overlay = label_overlay_grounding(rows=cached_doc_rows())
print(Counter((f.overlay, f.status) for f in overlay))
bad = [f for f in overlay if f.status == "ungrounded"]
for f in bad:
    print(f"UNSUPPORTED {f.overlay} {f.doc_id}.{f.field}: {f.value!r} ({f.reason})")
assert not bad, "overlay labels their evidence does not support; see data/labels/interim/README.md"

# Merged truth (Hub GT -> interim -> near-gold, as scoring merges it): where each scored field
# comes from, how the overlays overlap, and no overlay replacing a Hub label but a correction.
hub_rows = cached_doc_rows()
if hub_rows:
    merge = merged_truth(hub_rows.values())
    for doc_class, sources in merge.breakdown().items():
        print(doc_class, sources)
    for key, n in sorted(merge.overlaps.items(), key=lambda kv: -kv[1]):
        print(f"overlap {key}: {n}")
    # Hub labels an overlay disagrees with that no rule settled: the Hub value stands until a
    # review (or CUAD check) makes the overlay the clear winner, as an interim correction.
    print(f"{len(merge.hub_conflicts)} unsettled Hub conflicts")
    for c in merge.hub_conflicts:
        print(f"  {c.doc_id}.{c.field}: hub={c.hub_value!r} overlay={c.overlay_value!r} "
              f"grounded hub/overlay={c.hub_grounded}/{c.overlay_grounded}")
    assert not merge.violations, merge.violations[:5]


## 4. Agent gold sets

Every agent fixture keeps its own columns. The cleaner checks that each row has an `id`, parses `expected_fields` given as a JSON string, and attaches normalized fixture text when the row names a `subdir` and `filename`.

In [ ]:
from mailroom_sandbox.prep import clean_agent_rows

AGENTS = ["judge", "arbiter", "boss", "intake", "pdf_transcriber", "image_extractor"]
agent_reports = []
for agent in AGENTS:
    rows, report = clean_agent_rows(agent)
    entry = report.as_dict()
    entry["issues"] = len(report.issues)
    entry["with_text"] = sum(1 for r in rows if r.get("text"))
    agent_reports.append(entry)
    for issue in report.issues:
        print(f"[{report.source}] {issue}")

pd.DataFrame(agent_reports).set_index("source")

## 5. Write every prepared artifact

`prepare_offline_datasets()` reruns all cleaners and writes the JSONL files plus `prep_manifest.json`. `sandbox datasets prepare` runs the same function from the CLI.

In [ ]:
from mailroom_sandbox.prep import prepare_offline_datasets

summary = prepare_offline_datasets()
print(json.dumps({k: summary[k] for k in ("prepared_at", "fingerprint", "counts", "manifest")}, indent=2))

files = sorted(p for p in ctx.prepared_dir.rglob("*") if p.is_file())
pd.DataFrame(
    {"file": [str(p.relative_to(ctx.prepared_dir)) for p in files], "bytes": [p.stat().st_size for p in files]}
)

## 6. Validate what was written

`check_prepared_artifacts()` reads every file back from disk. It checks that the counts match the manifest, that ids are unique, that each fixture row has text and a class, that the balanced slice has exactly one row per fixture class, and that the fingerprint recomputes. A second prepare must give the **same fingerprint**, because the fingerprint ties eval records to dataset content.

In [ ]:
from mailroom_sandbox.notebook import check_prepared_artifacts

problems = check_prepared_artifacts(summary, ctx.root)
assert not problems, "\n".join(problems)

again = prepare_offline_datasets()
assert again["fingerprint"] == summary["fingerprint"], "prepare is not deterministic"
assert summary["counts"]["fixtures"] == fix_report.kept_rows
print(f"all checks passed; fingerprint {summary['fingerprint']} is stable across reruns")

## 7. Corpus overview

In [ ]:
by_class = (
    fixtures_df.groupby("expected_doc_class")["char_count"]
    .agg(docs="count", mean_chars="mean", min_chars="min", max_chars="max")
    .round(0)
    .astype(int)
    .sort_values("docs", ascending=False)
)
by_class

In [ ]:
import matplotlib.pyplot as plt

ordered = fixtures_df.sort_values("char_count")
fig, ax = plt.subplots(figsize=(8, 0.35 * len(ordered) + 1))
ax.barh(ordered["id"], ordered["char_count"], color="#4C78A8")
ax.set_xlabel("characters after cleaning")
ax.set_title("Prepared fixture length by document")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

**Next:** [`03_offline_sandbox_smoke.ipynb`](03_offline_sandbox_smoke.ipynb) runs mock evals against these files.